# Training & Evaluation - Classifying Spam Emails

Notebook n?y t?ng h?p ph?n train model spam/not spam cho project: ki?m tra d? li?u clean, th?ng k? nh?n, train 3 model c? b?n v? xem metric sau train.

Ghi ch?: notebook ?ang d?ng `SAMPLE_SIZE = 60000` ?? ch?y nhanh tr?n m?y c? nh?n. ??i th?nh `None` n?u mu?n train to?n b? `data/processed/combined_balanced_clean.csv`.

## 1. K?t qu? l?c d? li?u hi?n t?i

- Input rows: `296196`
- Clean rows: `294718`
- Removed rows: `1478`
- Label tr??c l?c: `{'0': 148098, '1': 148098}`
- Label sau l?c/c?n b?ng: `{'0': 147359, '1': 147359}`
- L?i ch?nh: `{'short_text': 1156}`

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "combined_balanced_clean.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

if not DATA_PATH.exists():
    from src.data_quality import clean_balanced_dataset

    print("Kh?ng th?y file clean, ?ang t?o l?i b?ng src.data_quality...")
    clean_balanced_dataset()

print("Project root:", PROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Data exists:", DATA_PATH.exists())

In [ ]:
dataset = pd.read_csv(DATA_PATH)
print(dataset.shape)
dataset.head()

In [ ]:
label_counts = dataset["label"].value_counts().sort_index().rename(index={0: "not spam", 1: "spam"})
label_percent = (label_counts / label_counts.sum() * 100).round(2)
pd.DataFrame({"count": label_counts, "percent": label_percent})

## 2. Ki?m tra naming/source code

- S? file Python ?? ki?m tra: `11`
- S? l?i ??t t?n: `0`
- Quy t?c: file/function/variable d?ng `snake_case`, constant d?ng `UPPER_CASE`, class d?ng `PascalCase`.

In [ ]:
style_report_path = REPORTS_DIR / "code_style_name_check.json"
style_report = json.loads(style_report_path.read_text(encoding="utf-8"))
style_report["naming_issues"]

## 3. Train model

Ba model ???c train c?ng pipeline `TfidfVectorizer + classifier`:

- `naive_bayes`: nhanh, baseline t?t cho text classification.
- `logistic_regression`: c?n b?ng gi?a t?c ??, ?? ch?nh x?c v? kh? n?ng gi?i th?ch.
- `linear_svm`: th??ng m?nh v?i TF-IDF v? d? li?u v?n b?n l?n.

Metric ?u ti?n l? `recall_spam` v? `f1_spam` v? b? s?t spam nguy hi?m h?n ch?n nh?m m?t s? email t?t.

In [ ]:
import sys
sys.path.append(str(PROJECT_ROOT))

from src.model_train import run_modeling_pipeline

SAMPLE_SIZE = 60000  # ??i th?nh None n?u mu?n train to?n b? dataset clean.
metrics_table = run_modeling_pipeline(sample_size=SAMPLE_SIZE)
metrics_table

## 4. Metric sau train g?n nh?t

| model               | accuracy | precision_spam | recall_spam | f1_spam  |
| ------------------- | -------- | -------------- | ----------- | -------- |
| linear_svm          | 0.979167 | 0.979007       | 0.979333    | 0.97917  |
| logistic_regression | 0.974    | 0.970394       | 0.977833    | 0.974099 |
| naive_bayes         | 0.963417 | 0.977339       | 0.948833    | 0.962875 |

In [ ]:
metrics_path = REPORTS_DIR / "model_metrics.csv"
metrics = pd.read_csv(metrics_path)
metrics.sort_values(["f1_spam", "recall_spam"], ascending=False)

In [ ]:
from src.model_evaluate import evaluate_from_predictions_csv

comparison = evaluate_from_predictions_csv(
    predictions_csv=REPORTS_DIR / "model_predictions.csv",
    true_col="label",
    pred_cols=["naive_bayes", "logistic_regression", "linear_svm"],
    output_dir=FIGURES_DIR,
)
comparison

## 5. Confusion matrix

C?c ?nh d??i ??y ???c sinh t? ??ng t? `src/model_evaluate.py`. N?u nh?m mu?n d?ng ?nh ngo?i cho b?o c?o cu?i, c? th? thay c?c file trong `reports/figures/`.

In [ ]:
from IPython.display import Image, display

for name in ["naive_bayes", "logistic_regression", "linear_svm"]:
    image_path = FIGURES_DIR / f"{name}_confusion_matrix.png"
    print(name, image_path)
    if image_path.exists():
        display(Image(filename=str(image_path)))

## 6. Demo predict

File `src/predict.py` load model t?t nh?t t? `models/spam_classifier.joblib` v? tr? v? nh?n `spam` ho?c `not spam`, k?m `spam_score` trong kho?ng 0-1.

In [ ]:
from src.predict import predict_email

sample_emails = [
    "Congratulations! You won a free iPhone. Click now to claim your prize.",
    "Hi team, please confirm your availability for the meeting tomorrow.",
    "Urgent account warning: verify your password immediately or your mailbox will be suspended.",
]

[predict_email(email) for email in sample_emails]

## 7. K?t lu?n nhanh

- Dataset clean hi?n c?n b?ng 2 class v? ?? lo?i d?ng qu? ng?n/l?i.
- `linear_svm` ?ang l? model t?t nh?t trong l?n train m?u 60,000 d?ng.
- Khi ch?t b?o c?o, nh?m c? th? train l?i v?i `SAMPLE_SIZE = None` ?? d?ng to?n b? dataset.